# R3b independent immutable-checkpoint analysis

This entry lists the already accepted Pilot checkpoints by default. It never starts
training, creates assets/probes or repeats the full Pilot. Formal500-step training
is not authorized. A bounded GPU diagnostic requires explicit operator selection
of ACTION=analyze-checkpoint; MAX_TASKS=1, exact QC Step3/revision0 Calibration
Action task. This is an engineering timing observation, not Development evidence.
Use existing HF_TOKEN/GITHUB_TOKEN; no Add Input or checkpoint downloads.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

EXPERIMENT = "CLM-001-R3b-POSTHOC"
ACTION = "list-checkpoints"  # Only lists saved checkpoints; no GPU work.

SOURCE_SHA = "216dd1d41ff2fb3b59ff340d9ab57c0d78eb62f8"
CODE = Path("/kaggle/working/minicells-r3b-analysis-code-" + SOURCE_SHA[:12])
WORK = Path("/kaggle/working/minicells-r3b-analysis-" + SOURCE_SHA[:12])
ENTRY = (
    CODE
    / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/analyze.py"
)
os.environ.update(
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)


def execute(args, cwd=None):
    print("[R3b] " + " ".join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True)


secrets = UserSecretsClient()
for name in (("HF_TOKEN", "GITHUB_TOKEN") if ACTION != "list-checkpoints" else ("HF_TOKEN",)):
    if not os.environ.get(name, "").strip():
        try:
            os.environ[name] = secrets.get_secret(name).strip()
        except Exception:
            raise RuntimeError("Enable existing Kaggle Secret " + name) from None
    if not os.environ[name]:
        raise RuntimeError("Missing existing Kaggle Secret " + name)

if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=False)
    execute(["git", "init", "--quiet", CODE])
    execute(["git", "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git"], CODE)
    execute(["git", "sparse-checkout", "init", "--no-cone"], CODE)
    execute(
        [
            "git",
            "sparse-checkout",
            "set",
            "--no-cone",
            "/pyproject.toml",
            "/src/",
            "/shared/python/",
            "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
            "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
            "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
            "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
            "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
            "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
        ],
        CODE,
    )
    execute(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], CODE)
    execute(["git", "checkout", "--detach", "FETCH_HEAD"], CODE)
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE, text=True).strip() != SOURCE_SHA:
    raise RuntimeError("SOURCE_SHA_MISMATCH")

# Install only missing data/storage packages. Torch/numpy are supplied by Kaggle.
requirements = {
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface_hub>=0.23",
}
missing = [
    requirement
    for module, requirement in requirements.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    execute([sys.executable, "-m", "pip", "install", "--quiet", *missing])
baseline = json.loads((ENTRY.parent / "reports/diagnostic-optimization/baseline-evidence.json").read_text())
reference = baseline["evidence"]["checkpoint-index.json"]["checkpoint_ref"]
WORK.mkdir(parents=True, exist_ok=True)
ref_file = WORK / "pinned-input-reference.json"
ref_file.write_text(json.dumps(reference))
print({"TRAINING_SOURCE_SHA": baseline["training_source_sha"], "ANALYSIS_SOURCE_SHA": SOURCE_SHA,
       "hf_repo": reference["hf_repo"], "hf_revision": reference["hf_revision"],
       "ACTION": ACTION, "max_tasks": 1, "formal_training_authorized": False}, flush=True)
args = [sys.executable, ENTRY, "--mode", ACTION, "--checkpoint-ref", ref_file,
        "--output", WORK / "analysis", "--max-tasks", "1",
        "--arm", "QC-R3b", "--step", "3", "--event-revision", "0",
        "--stage", "action", "--role", "calibration", "--phase", "post_event",
        "--device", "cuda:0" if ACTION == "analyze-checkpoint" else "cpu",
        ]
if ACTION != "list-checkpoints":
    args += ["--publish-github"]
execute(args, CODE)
